# House-Edge — a quantitative teardown 🔬
### Margin vs futures/CFD funding · the zero-markup identity · excess-Sharpe accounting · the markup sweep · where the dream dies

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Free leverage?: Busted](https://img.shields.io/badge/Free_leverage%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim with its number.* The steelman: a vol-targeted contrarian dip-buyer with a trend gate, levered, beats buy-and-hold. We finance it through the two account types that exist in the real world — each charging every dollar exactly once — and show the drawdown protection is real while the market-beating never materialises, with the broker's markup as the dial that turns a near-tie into a structural loss.

> ⚠️ **Not investment advice.** The core executes on a synthetic GARCH-with-bear-regimes control; the real S&P run is in [`../docs/results.md`](../docs/results.md), sources in [`../docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back to intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from house_edge import data, strategy, costs, extension

# Offline synthetic control: an equity index with GARCH vol-clustering + persistent bear regimes.
# The real S&P 500 verdict is in ../docs/results.md.
price, rate, truth = data.synthetic_market(seed=30)
e = strategy.exposure(price)
print(f"synthetic control: {truth.n_days} days, {truth.n_crashes} bear regimes, seed {truth.seed}")


synthetic control: 6048 days, 9 bear regimes, seed 30


## Beat 0 · Verdict

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — does the risk control work? | 🟢 `REAL` | maxDD **-24.7%** vs buy-and-hold **-55.7%** (S&P, 1990-2026), Calmar **0.38** vs **0.19**; robust on the synthetic control. |
| **Tradability** — beats buy-and-hold once financed honestly? | 🔴 `MIRAGE` | Never out-earns it: funded flat **10.1%** vs **10.6%**; at the retail CFD markup **7.4%** at excess Sharpe **0.38** vs **0.50**. |
| **Free leverage?** | ⚪ `Busted` | The CFD markup lands on the FULL notional: **2.65 pts/yr** vs the margin account's **0.66** at the same 2.5%. |

> **In one sentence:** the book halves the drawdown and, funded at the fair bill rate, nearly matches the index — but it never beats it, and the retail pitch dies at the broker's markup, which a CFD charges on money you never borrowed.

*(This notebook executes on the synthetic control; the real numbers are in [`../docs/results.md`](../docs/results.md).)*

## Beat 1 · The claim, precisely

Exposure $e_t = \mathrm{clip}(\sigma^\star/\hat\sigma_t,\,0,\,2)\cdot m_t\cdot g_t$, where $\sigma^\star$ is the target vol, $\hat\sigma_t$ trailing 20-day realised vol, $m_t\in\{1.3,1,0.6\}$ the RSI(2) contrarian multiplier, and $g_t\in\{0,1\}$ the 200-day trend gate. Net daily returns under the two account types (with markup $m$, dividend yield $q$, bill $r^f$):

$$r^{\text{margin}}_t = e\,(r_t + q) - (e-1)^+(r^f + m) + (1-e)^+\,r^f - \text{spread}\,|\Delta e|$$
$$r^{\text{futures}}_t = r^f + e\,(r_t + q - r^f - m) - \text{spread}\,|\Delta e|$$

**The identity:** at $m=0$ both reduce to $e\,(r_t+q) + (1-e)\,r^f$ — funding itself is symmetric; the models differ only in where the markup lands ($(e-1)^+$ vs $e$). The claim under test: $\mathrm{CAGR}(r^{\text{margin}}) > \mathrm{CAGR}(\text{buy\&hold})$.

In [2]:
# the zero-markup identity, verified numerically
a = costs.net_returns(e, price, rate, mode='margin', markup=0.0)
b = costs.net_returns(e, price, rate, mode='futures', markup=0.0)
print(f'margin == futures at markup 0: {np.allclose(a.to_numpy(), b.to_numpy())}')
s_bh = strategy.summary(costs.buy_and_hold(price), rf=rate)
s_mg = strategy.summary(costs.net_returns(e, price, rate, mode='margin'), rf=rate)
print(f"synthetic: strat CAGR {s_mg['cagr']*100:.1f}% vs buy&hold {s_bh['cagr']*100:.1f}% "
      f"-- claim is {'TRUE' if s_mg['cagr']>s_bh['cagr'] else 'FALSE'} here too")

margin == futures at markup 0: True
synthetic: strat CAGR 6.1% vs buy&hold 7.8% -- claim is FALSE here too


## Beat 2 · So what?

Two literatures collide. Vol-targeting (Moreira-Muir 2017) and short-horizon reversal (Lehmann 1990) are real and raise *risk-adjusted* returns. Against them: the cost-of-carry (Hull) prices a synthetic position at the bill plus the broker's markup, constant leverage $k$ pays a volatility tax $\approx \tfrac12(k^2-k)\sigma^2$, and every day the trend gate holds you in cash forgoes the equity premium. The empirical question is what's left at each markup — and the accounting trap to avoid is double-charging: a model that finances the full notional *and* credits cash interest only on the un-deployed fraction pays $r^f$ twice and manufactures a phantom $\bar e \cdot r^f \approx 2.7$ pts/yr drag (an earlier version of this study did exactly that).

## Beat 3 · Pre-registered protocol

1. **Drawdown** (`extension.drawdown_protection`): strat maxDD vs buy-and-hold. Real ⇔ strictly smaller.
2. **Return** (`strategy.summary(..., rf=rate)` on `costs.net_returns`): CAGR vs buy-and-hold, Sharpe in **excess of the bill** on both sides — the only convention that doesn't hand buy-and-hold the risk-free rate for free.
3. **House edge** (`costs.house_edge`): the markup bill by account type — CFD (full notional) vs margin (borrowed slice).
4. **Robustness** (`extension.financing_sweep`): edge vs buy-and-hold across the markup, both account types.

**Mirage line:** the book never out-earns buy-and-hold, and at retail markups (1.5–3% over the bill) it clearly loses.

## Beat 4 · The teardown

### 4a · Drawdown protection (real) vs the return tie-at-best (mirage)

In [3]:
rows = {'buy & hold':        strategy.summary(costs.buy_and_hold(price), rf=rate),
        'funded flat (0%)':  strategy.summary(costs.net_returns(e, price, rate, mode='futures', markup=0.0), rf=rate),
        'margin @ 2.5%':     strategy.summary(costs.net_returns(e, price, rate, mode='margin'), rf=rate),
        'CFD @ 2.5%':        strategy.summary(costs.net_returns(e, price, rate, mode='futures'), rf=rate)}
tbl = pd.DataFrame(rows).T[['cagr','sharpe','max_drawdown','calmar']]
display((tbl*[100,1,100,1]).round(2).rename(columns={'cagr':'CAGR%','sharpe':'Sharpe(excess)','max_drawdown':'maxDD%'}))

,CAGR%,Sharpe(excess),maxDD%,calmar
buy & hold,7.800,0.280,-44.270,0.180
funded flat (0%),6.370,0.220,-26.740,0.240
margin @ 2.5%,6.080,0.200,-26.940,0.230
CFD @ 2.5%,4.520,0.090,-29.140,0.160


> 💡 **In plain words.** The drawdown shrinks a lot (real risk control) and the return shrinks a little — *until the CFD markup hits*, when it shrinks a lot. On the real S&P: funded flat **10.1%** vs **10.6%** with a better excess Sharpe (0.54 vs 0.50) and double the Calmar (0.41 vs 0.19); the CFD takes it to **7.4%**. The strategy is fine — the retail funding isn't.

### 4b · The house-edge identity — where the markup lands

In [4]:
he = costs.house_edge(e, price, rate)
print(f"funded flat {he['cagr_funded_flat']*100:.2f}%  |  margin @2.5% {he['cagr_margin']*100:.2f}%  "
      f"|  CFD @2.5% {he['cagr_cfd']*100:.2f}%")
print(f"house edge (flat - CFD): {he['house_edge_ann']*100:.2f} pts/yr  ~=  avg exposure "
      f"{he['avg_exposure']:.2f} x 2.5% markup  |  margin pays only {he['margin_edge_ann']*100:.2f}")
print('On the real S&P (../docs/results.md): CFD bill 2.65 pts/yr vs margin 0.66, avg exposure 0.97x.')

funded flat 6.37%  |  margin @2.5% 6.08%  |  CFD @2.5% 4.52%
house edge (flat - CFD): 1.85 pts/yr  ~=  avg exposure 0.70 x 2.5% markup  |  margin pays only 0.28
On the real S&P (../docs/results.md): CFD bill 2.65 pts/yr vs margin 0.66, avg exposure 0.97x.


> 💡 **In plain words.** Both accounts borrow money at the same fair rate. The difference is the *markup*: a margin account pays it on the slice above your capital (here, rarely much), a CFD pays it on the **whole position** — including the first 100% you never borrowed. That ≈ avg-exposure × markup is the house's rent, and it's the entire gap between the two columns.

## Beat 5 · The verdict

- **Real risk control** (4a): maxDD -24.7% vs -55.7%, Calmar 0.38 vs 0.19.
- **No return edge anywhere** (4a): funded flat 10.1% vs 10.6% — a tie at best; CFD @2.5% 7.4%, excess Sharpe 0.38 vs 0.50.
- **The house edge** (4b): the CFD markup bill is 2.65 pts/yr vs the margin account's 0.66 — markup charged on money never borrowed.

> **Signal `REAL` · Tradability `MIRAGE` · Free leverage? `Busted`.** The mirage is the retail markup, not financing per se — the exact line where the dream dies.

## Beat 6 · Could you trade it?

- **Not for the advertised reason:** the CAGR edge vs buy-and-hold is negative at *every* markup, in *both* account types — even at zero. Volatility drag plus the gate's time-in-cash see to that before the broker takes anything.
- **The venue decides the price of the insurance.** Futures (near-flat funding): half the drawdown for ~0.5 pts/yr — a defensible institutional overlay. Retail CFD at 1.5–3%: the same insurance costs 2–3 pts/yr because the markup hits the full notional. Same book, same signal.
- **Honest use case:** a drawdown-reduction overlay for capital that cannot survive −56% — sized for survival, funded as close to flat as your access allows, and never sold as a market-beater.

## Beat 7 · Going further

### 7a · Worked complement — the markup sweep, both account types
Is the shortfall a fee artefact? Sweep the broker markup over the bill and watch where the dream dies.

In [5]:
sw = extension.financing_sweep(price, rate)
display((sw*[100,1,100,100,1,100]).round(2)
        .rename(columns={'margin_cagr':'margin_CAGR%','margin_sharpe':'margin_Sharpe',
                         'margin_edge_cagr':'margin_edge%','cfd_cagr':'cfd_CAGR%',
                         'cfd_sharpe':'cfd_Sharpe','cfd_edge_cagr':'cfd_edge%'}))
print('Real S&P (../docs/results.md): edge -0.5 pt at markup 0 (a rounding error), '
      '-2.1 (CFD @1.5%) to -3.2 (CFD @2.5%) at retail markups.')

,margin_CAGR%,margin_Sharpe,margin_edge%,cfd_CAGR%,cfd_Sharpe,cfd_edge%
markup,,,,,,
0.000,6.370,0.220,-1.430,6.370,0.220,-1.430
0.010,6.260,0.220,-1.550,5.630,0.170,-2.180
0.015,6.200,0.210,-1.600,5.260,0.140,-2.550
0.025,6.080,0.200,-1.720,4.520,0.090,-3.280
0.050,5.800,0.180,-2.000,2.700,-0.050,-5.100


Real S&P (../docs/results.md): edge -0.5 pt at markup 0 (a rounding error), -2.1 (CFD @1.5%) to -3.2 (CFD @2.5%) at retail markups.


**The result.** The edge over buy-and-hold never goes positive — but the *story* is in how it dies. At markup 0 the two models coincide and the book essentially ties the index (**-0.5 pt** on the real tape) with half the drawdown. The margin account decays gently (markup × borrowed slice). The CFD decays at ≈ markup × *full* exposure — at the 1.5–3% retail swaps, 2–3 pts/yr gone. The mirage isn't leverage and it isn't financing; it's the **house's markup on the notional**. Full real run in [`../docs/results.md`](../docs/results.md).

### 7b · Other forks
- **Levered constant-tilt vs timing overlay** — does a static 1.3× risk-parity book (no timing) dominate the vol-targeted timing one, net of the same honest carry?
- **Utility-priced drawdown** — at what risk aversion does −25% vs −56% justify ~1 pt/yr of insurance premium? Make the trade-off explicit.
- **Term-structure of the markup** — retail CFD swaps run 1.5–3.5% over the bill; institutional futures roll within ~0.3%. Price the same book across the access spectrum.

PRs welcome — add the constant-tilt benchmark or the utility model.